In [1]:
import pandas as pd
from datasets import load_from_disk

In [5]:
halu = load_from_disk("../datasets/HaluEval_QA")

truthfulqa = load_from_disk("../datasets/TruthfulQA")

mnli = load_from_disk("../datasets/MNLI")

custom = load_from_disk("../datasets/CustomFactDataset")

In [6]:
texts = []
labels = []
sources = []

## HaluEval Processing

In [7]:
for sample in halu["data"]:

    question = sample["question"]

    right_answer = sample["right_answer"]

    hallucinated_answer = sample["hallucinated_answer"]

    # factual sample
    factual_text = question + " " + right_answer

    texts.append(factual_text)

    labels.append(0)

    sources.append("HaluEval")

    # hallucinated sample
    hallucinated_text = question + " " + hallucinated_answer

    texts.append(hallucinated_text)

    labels.append(1)

    sources.append("HaluEval")

## MNLI PROCESSING

In [8]:
label_map = {
    0: 0,   # entailment -> factual
    2: 1    # contradiction -> hallucination
}

In [9]:
for sample in mnli["train"]:

    if sample["label"] not in [0, 2]:
        continue

    premise = sample["premise"]

    hypothesis = sample["hypothesis"]

    combined_text = premise + " " + hypothesis

    label = label_map[sample["label"]]

    texts.append(combined_text)

    labels.append(label)

    sources.append("MNLI")

## CUSTOM DATASET PROCESSING

In [10]:
for sample in custom:

    claim = str(sample["claim"])

    label_text = str(sample["label"]).lower()

    # SUPPORTS -> factual
    if label_text == "supports":

        label = 0

    # REFUTES -> hallucination
    elif label_text == "refutes":

        label = 1

    else:
        continue

    texts.append(claim)

    labels.append(label)

    sources.append("Custom")

## CREATE FINAL DATAFRAME

In [11]:
df = pd.DataFrame({
    "text": texts,
    "label": labels,
    "source": sources
})

In [12]:
print(df.head())

                                                text  label    source
0  Which magazine was started first Arthur's Maga...      0  HaluEval
1  Which magazine was started first Arthur's Maga...      1  HaluEval
2  The Oberoi family is part of a hotel company t...      0  HaluEval
3  The Oberoi family is part of a hotel company t...      1  HaluEval
4  Musician and satirist Allie Goertz wrote a son...      0  HaluEval


In [13]:
print(df["label"].value_counts())

label
0    220934
1    170678
Name: count, dtype: int64


In [15]:
df.to_csv(
    "../processed/final_dataset.csv",
    index=False
)